In [5]:
from sarma.ingestion.knowledge_base import load_knowledge_base
from sarma.vectorstore.vectorstore import create_vector_store
from sarma.vectorstore.vectorstore import load_vector_store
from sarma.ingestion.splitter import split_documents
from sarma.retriever import create_retriever
from sarma.rag.rag import create_rag_chain
from sarma.prompts import rag_prompt
from sarma.llm import llm
import sarma.assistant as sas
from sarma.graph.workflow import create_sarma_graph

In [6]:
try:
    db = load_vector_store()
    print("Loaded existing vector database")
    
except FileNotFoundError:
    print("Creating vector database...")
    #documents = load_pdf("../data/raw/RB209 Arable crops.pdf")
    documents = load_knowledge_base("../data/knowledge_base")
    chunks = split_documents(documents)
    db = create_vector_store(chunks)
    
retriever = create_retriever(db)
chain = create_rag_chain(retriever)

Loaded existing vector database


In [7]:
response = chain.invoke("What is the best Nitrogen value for wheat?")
print(response.content)

The best nitrogen (N) value for wheat depends on the type of wheat:  
- **Feed wheat** has an economic optimum nitrogen concentration of **1.9% N** (corresponding to 11% protein).  
- **Bread-making wheat** has an economic optimum nitrogen concentration of **2.1% N** (corresponding to 12% protein).  

These values are based on the economic optimum rate of nitrogen for each wheat type.


In [8]:
results = db.similarity_search_with_score(
"What is the best Nitrogen value for wheat?",
k=10
)

for doc, score in results:
    print('SCORE:', score)
    print(doc.page_content[:200])
    print("="*50)

SCORE: 0.3302614688873291
Return to Contents
Wheat – use of grain nitrogen concentration
Farm nitrogen strategies for wheat can be assessed periodically using information 
on grain protein concentration. Grain protein at the e
SCORE: 0.3302614688873291
Return to Contents
Wheat – use of grain nitrogen concentration
Farm nitrogen strategies for wheat can be assessed periodically using information 
on grain protein concentration. Grain protein at the e
SCORE: 0.3302615284919739
Return to Contents
Wheat – use of grain nitrogen concentration
Farm nitrogen strategies for wheat can be assessed periodically using information 
on grain protein concentration. Grain protein at the e
SCORE: 0.3302615284919739
Return to Contents
Wheat – use of grain nitrogen concentration
Farm nitrogen strategies for wheat can be assessed periodically using information 
on grain protein concentration. Grain protein at the e
SCORE: 0.3302615284919739
Return to Contents
Wheat – use of grain nitrogen concentration
Far

In [9]:
print(db._collection.count())

3442


In [10]:
items = db._collection.get(
    limit=10,
    include=["metadatas", "documents"]
)

for meta in items["metadatas"]:
    print(meta)

{'creationdate': '2017-12-06T09:57:02+00:00', 'trapped': '/False', 'page': 1, 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'total_pages': 52, 'moddate': '2017-12-06T09:57:17+00:00', 'source': 'RB209 Arable crops.pdf', 'page_label': '4S1', 'producer': 'Adobe PDF Library 15.0'}
{'moddate': '2017-12-06T09:57:17+00:00', 'total_pages': 52, 'source': 'RB209 Arable crops.pdf', 'page': 3, 'producer': 'Adobe PDF Library 15.0', 'creationdate': '2017-12-06T09:57:02+00:00', 'page_label': '4S3', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'trapped': '/False'}
{'creationdate': '2017-12-06T09:57:02+00:00', 'producer': 'Adobe PDF Library 15.0', 'total_pages': 52, 'page': 3, 'source': 'RB209 Arable crops.pdf', 'page_label': '4S3', 'moddate': '2017-12-06T09:57:17+00:00', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'trapped': '/False'}
{'source': 'RB209 Arable crops.pdf', 'moddate': '2017-12-06T09:57:17+00:00', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'page_label': '4S3', 'trapped':

In [13]:
response1 = chain.invoke(
    "What nitrogen rate is recommended for spring sown wheat?"
)

print(response1.content)

The recommended nitrogen rates for spring sown wheat depend on soil type, as outlined in **Table 4.18**:

- **Light sand soils**: 160, 130, 100, 70, 40, 0–40, 0 kg N/ha  
- **All other mineral soils**: 210a, 180, 150, 120, 70, 40, 0–40 kg N/ha  
- **Organic soils**: 120, 70, 40, 0–40 kg N/ha  
- **Peaty soils**: 0–40 kg N/ha  

Note: The "a" in 210a indicates the recommendation exceeds the nitrogen maximum limit in Nitrate Vulnerable Zones (NVZs), which applies to the entire farm area for a crop type, not individual fields. For detailed guidance, refer to [www.gov.uk/nitrate-vulnerable-zones](https://www.gov.uk/nitrate-vulnerable-zones).
